In [20]:
import numpy as np
import random

I used recursion to check which player could force a win.
Empty cells are represented by `-1`, and the function returns
`True` if $P_0$ can force a win with the indicated player to move.

In [21]:
def winning(M, player):
    n = len(M)
    empty_positions = [(i, j) for i in range(n)
                       for j in range(n) if M[i, j] == -1]
    if len(empty_positions) == 0:
        return round(np.linalg.det(M)) == 0
    if player == 0:
        for i, j in empty_positions:
            new_M = M.copy()
            new_M[i, j] = 0
            if winning(new_M, 1):
                return True
        return False
    else:
        for i, j in empty_positions:
            new_M = M.copy()
            new_M[i, j] = 1
            if not winning(new_M, 0):
                return False
        return True


$P_0$ can force a win for $n=3$ regardless of who starts.
This is the result explained mathematically in Claim 3.

In [22]:

n = 3
for player in [0, 1]:
    M = np.full((n, n), -1)
    if winning(M, player):
        print("P", player, "starts: P0 wins")
    else:
        print("P", player, "starts: P1 wins")


P 0 starts: P0 wins
P 1 starts: P0 wins


## Testing the pairing strategy

I test the strategy for $n=4$ and $n=5$, with both starting
players. My opponent chooses randomly, completes my unfinished
pair (`block`), or opens an empty pair (`avoid`), where possible.

Each combination is tested over $500$ games.

In [23]:
def make_pairs(n, first, first_one=None):
    if n % 2 == 1:
        if first == 0:
            special = (n - 1, n - 1)
        else:
            special = first_one
        rows = [i for i in range(n) if i != special[0]][:4]
    else:
        special = None
        rows = [0, 1, 2, 3]
    pairs = []
    for j in range(n):
        pairs.append(((rows[0], j), (rows[1], j)))
        pairs.append(((rows[2], j), (rows[3], j)))
    other = [(i, j) for i in range(n) for j in range(n)
             if i not in rows and (i, j) != special]
    for k in range(0, len(other), 2):
        pairs.append((other[k], other[k + 1]))
    partner = {}
    for a, b in pairs:
        partner[a] = b
        partner[b] = a

    return pairs, partner, rows, special
def choose_zero(M, partner):
    for a, b in partner.items():
        if M[a] == 1 and M[b] == -1:
            return b
    for a, b in partner.items():
        if M[a] == -1 and M[b] == -1:
            return a
    raise ValueError("No legal move for P0")

def choose_one(M, partner, strategy):
    empty = [a for a in partner if M[a] == -1]
    if strategy == "block":
        choices = [a for a in empty if M[partner[a]] == 0]
    elif strategy == "avoid":
        choices = [a for a in empty if M[partner[a]] == -1]
    else:
        choices = empty
    return random.choice(choices if choices else empty)

def play_game(n, first, opponent):
    M = np.full((n, n), -1, dtype=int)
    if n % 2 == 1 and first == 1:
        first_one = (random.randrange(n), random.randrange(n))
    else:
        first_one = None
    pairs, partner, rows, special = make_pairs(
        n, first, first_one
    )
    if special is not None:
        M[special] = first
        player = 1 - first
    else:
        player = first
    while np.any(M == -1):
        if player == 0:
            move = choose_zero(M, partner)
        else:
            move = choose_one(M, partner, opponent)

        M[move] = player
        player = 1 - player
    paired = all(M[a] + M[b] == 1 for a, b in pairs)
    row_relation = np.array_equal(
        M[rows[0]] + M[rows[1]],
        M[rows[2]] + M[rows[3]]
    )
    zero_det = np.isclose(np.linalg.det(M), 0)
    return paired and row_relation and zero_det, M, rows


In [24]:
for n in [4, 5]:
    for first in [0, 1]:
        for opponent in ["random", "block", "avoid"]:
            wins = 0
            for trial in range(500):
                success, M, rows = play_game(n, first, opponent)
                if success:
                    wins += 1
            print(f"n={n}, first=P{first}, opponent={opponent}: "
                  f"{wins}/500 successful")

n=4, first=P0, opponent=random: 500/500 successful
n=4, first=P0, opponent=block: 500/500 successful
n=4, first=P0, opponent=avoid: 500/500 successful
n=4, first=P1, opponent=random: 500/500 successful
n=4, first=P1, opponent=block: 500/500 successful
n=4, first=P1, opponent=avoid: 500/500 successful
n=5, first=P0, opponent=random: 500/500 successful
n=5, first=P0, opponent=block: 500/500 successful
n=5, first=P0, opponent=avoid: 500/500 successful
n=5, first=P1, opponent=random: 500/500 successful
n=5, first=P1, opponent=block: 500/500 successful
n=5, first=P1, opponent=avoid: 500/500 successful


All $6000$ trials passed the pairing, row relation and
determinant checks. These results agree with the proofs,
which explain why the strategy works against every opponent.